# GRPO v1 Reinforcement Learning:

`google/translategemma-4b-it`
## Spanish → Catalan (Valencian Variety)

**Pipeline overview:**

1. Install dependencies
2. Load `google/translategemma-4b-it` with 4-bit quantization + LoRA (starting from SFT checkpoint)
3. **GRPO** with a composite reward:
   - `r_c` — sentence-level chrF (fidelity to reference)
   - `r_t` — p(HT | text) from a translationese classifier (HT vs. MT)
   - `r_final = (1 − α) · r_c + α · r_t` with linear warm-up on α
4. Save and push GRPO model to Hugging Face Hub
Available at HuggingFace: `guerreropaula/80translategemma4b-grpo-es-va`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 1. Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    sacrebleu \
    sentencepiece \
    huggingface_hub \
    flash-attn-triton

In [ ]:
%%capture
!pip install -q git+https://github.com/google-research/bleurt.git
!wget -q https://storage.googleapis.com/bleurt-oss/bleurt-base-128.zip
!unzip -q bleurt-base-128.zip

In [ ]:
import torch
import transformers, peft, trl

print(f"PyTorch      : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch      : 2.10.0+cu128
transformers : 5.0.0
peft         : 0.18.1
trl          : 0.29.0
CUDA         : True
GPU          : NVIDIA A100-SXM4-80GB
VRAM         : 85.1 GB


---
## 2. Hugging Face Login

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""
login(token=HF_TOKEN)

---
## 3. Load SFT Checkpoint: `guerreropaula/translategemma4b-sft-es-va2`

GRPO training starts from the SFT-fine-tuned model (already on Hugging Face Hub),
loaded with 4-bit quantization so it fits in a single GPU.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

SFT_MODEL_ID   = "guerreropaula/translategemma4b-sft-es-va2"
BASE_MODEL_ID  = "google/translategemma-4b-it"
MAX_SEQ_LENGTH = 256
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16       = torch.cuda.is_bf16_supported()

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16 if USE_BF16 else torch.float16,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model quantized, then re-attach the SFT LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = HF_TOKEN,
    torch_dtype         = torch.bfloat16 if USE_BF16 else torch.float16,
    trust_remote_code   = True,
)
base_model = prepare_model_for_kbit_training(base_model)

# Load the SFT LoRA adapter on top
model = PeftModel.from_pretrained(base_model, SFT_MODEL_ID, is_trainable=True, token=HF_TOKEN)
model.print_trainable_parameters()

print(f"Base model : {BASE_MODEL_ID}")
print(f"SFT adapter: {SFT_MODEL_ID}")


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model  : google/translategemma-4b-it
Params : 2490M


---
## 4. TranslateGemma Prompt Template

TranslateGemma uses a structured chat format where each user-turn message
includes the fields `type`, `source_lang_code`, `target_lang_code`, and `text`.
The model uses `"ca"` as the language code for Catalan/Valencian (no dialect differentiation).

In [ ]:
SOURCE_LANG_CODE = "es"
TARGET_LANG_CODE = "ca"   # TranslateGemma does not differentiate Valencian from Catalan
SOURCE_COL       = "ES"
TARGET_COL       = "VA"


def _make_messages(source_text: str) -> list:
    """Build the user-turn message list for the TranslateGemma template."""
    return [
        {
            "role": "user",
            "content": [
                {
                    "type"             : "text",
                    "source_lang_code" : SOURCE_LANG_CODE,
                    "target_lang_code" : TARGET_LANG_CODE,
                    "text"             : source_text,
                }
            ],
        }
    ]


def format_for_sft(source_text: str, target_text: str) -> str:
    """Full prompt + reference answer; used to build the SFT dataset."""
    prompt = tokenizer.apply_chat_template(
        _make_messages(source_text), tokenize=False, add_generation_prompt=True
    )
    return prompt + target_text + tokenizer.eos_token


def make_inference_prompt(source_text: str) -> str:
    """Prompt only (no answer); used for inference and GRPO."""
    return tokenizer.apply_chat_template(
        _make_messages(source_text), tokenize=False, add_generation_prompt=True
    )


# Sanity check
test_src = "El ayuntamiento ha aprobado el nuevo presupuesto."
test_tgt = "L'ajuntament ha aprovat el nou pressupost."
print("=== SFT format ===")
print(format_for_sft(test_src, test_tgt))
print("\n=== Inference / GRPO format ===")
print(make_inference_prompt(test_src))

=== SFT format ===
<bos><start_of_turn>user
You are a professional Spanish (es) to Catalan (ca) translator. Your goal is to accurately convey the meaning and nuances of the original Spanish text while adhering to Catalan grammar, vocabulary, and cultural sensitivities.
Produce only the Catalan translation, without any additional explanations or commentary. Please translate the following Spanish text into Catalan:


El ayuntamiento ha aprobado el nuevo presupuesto.<end_of_turn>
<start_of_turn>model
L'ajuntament ha aprovat el nou pressupost.<eos>

=== Inference / GRPO format ===
<bos><start_of_turn>user
You are a professional Spanish (es) to Catalan (ca) translator. Your goal is to accurately convey the meaning and nuances of the original Spanish text while adhering to Catalan grammar, vocabulary, and cultural sensitivities.
Produce only the Catalan translation, without any additional explanations or commentary. Please translate the following Spanish text into Catalan:


El ayuntamiento 

---
## 5. Training Dataset: `gplsi/amic_parallel`

This dataset is also used to build the GRPO training prompts (see section 8).


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("gplsi/amic_parallel")
print(raw_dataset)
print("\nExample:", raw_dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

data/train/va-es-documents.curated.jsonl:   0%|          | 0.00/404M [00:00<?, ?B/s]

data/train/va-es-paragraph.length.curate(…):   0%|          | 0.00/245M [00:00<?, ?B/s]

data/train/va-es-sentence.length.ner.cur(…):   0%|          | 0.00/91.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/680978 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['VA', 'ES', 'source'],
        num_rows: 680978
    })
})

Example: {'VA': "Música a la UPV\n\nDel pasdoble 'Agüero' de José Franco, al 'Bolero' de Ravel, gaudeix el divendres 11 del concert de la Banda Simfònica de la UPV a l'Alfons Roig (19.30 h)\n\nL'Auditori Alfons Roig de la Facultat de Belles Arts (edifici 3N, planta baixa) de la Universitat Politècnica de València (UPV) és l'escenari elegit per a la celebració, el pròxim divendres, 11 de març, d'un nou concert de la Banda Simfònica de la UPV.\nAmb accés lliure fins a completar l'aforament del recinte, l'agrupació musical universitària dirigida per Francisco J. Valero interpretarà, a partir de les 19.30 hores, un programa dividit en dues parts.\n\nObres de José Franco, Philip Wilby, Josep Alamà Gil, George Bizet i Maurice Ravel\n\nLa primera començarà a ritme de pasdoble, amb Agüero, de José Franco, peça a la qual seguiran el Concert per a bombardí de Philip Wilby -amb Jorge L

---
## 6. GRPO Training Dataset


In [ ]:
GRPO_TRAIN_SAMPLES = 5_000
GRPO_OUTPUT_DIR    = "./translategemma4b_grpo_es_va"


def make_grpo_example(examples):
    return {
        "prompt"   : [make_inference_prompt(src) for src in examples[SOURCE_COL]],
        "reference": list(examples[TARGET_COL]),
    }


grpo_raw     = raw_dataset["train"].shuffle(seed=123).select(range(GRPO_TRAIN_SAMPLES))
grpo_dataset = grpo_raw.map(
    make_grpo_example,
    batched        = True,
    remove_columns = grpo_raw.column_names,
)

print("GRPO dataset:", grpo_dataset)
print("\nPrompt example:")
print(grpo_dataset[0]["prompt"][:250], "...")
print("Reference:", grpo_dataset[0]["reference"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

GRPO dataset: Dataset({
    features: ['prompt', 'reference'],
    num_rows: 5000
})

Prompt example:
<bos><start_of_turn>user
You are a professional Spanish (es) to Catalan (ca) translator. Your goal is to accurately convey the meaning and nuances of the original Spanish text while adhering to Catalan grammar, vocabulary, and cultural sensitivities. ...
Reference: Se celebrarà a partir de les 19:00 h del dia 19 de novembre en el Paranimf.


---
## 7. Composite Reward Function with Linear Warm-Up

**Two components:**
- `r_c` — sentence-level chrF normalized to [0, 1]; measures fidelity to the reference
- `r_t` — p(HT | text) from `guerreropaula/ht_mt_classifier_best`; measures naturalness

**Combined reward with warm-up:**

$$r_{\text{final}} = (1 - \alpha) \cdot r_c + \alpha \cdot r_t$$

where α increases linearly from 0 to `CLF_WEIGHT_MAX` after the first `CLF_WARMUP_STEPS` steps.
During warm-up the classifier is not called, reducing VRAM pressure while the model
first learns to produce valid Valencian translations.

In [ ]:
import sacrebleu
import torch.nn.functional as F
from typing import List
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

CLF_REPO_ID  = "guerreropaula/ht_mt_classifier_best"
HT_LABEL_IDX = 1   # label mapping: 0 = MT,  1 = HT

_clf_tok   = AutoTokenizer.from_pretrained(CLF_REPO_ID)
_clf_model = AutoModelForSequenceClassification.from_pretrained(CLF_REPO_ID)

_clf_model.eval().to(DEVICE)

print(f"Classifier : {CLF_REPO_ID}")
print(f"Labels     : {_clf_model.config.id2label}")

config.json:   0%|          | 0.00/870 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classifier : guerreropaula/ht_mt_classifier_best
Labels     : {0: 'MT', 1: 'HT'}


In [ ]:
CLF_WARMUP_STEPS = 50    # steps with α = 0 (chrF only)
CLF_WEIGHT_MAX   = 0.3   # maximum classifier weight after warm-up
TOTAL_STEPS      = 200   # must match max_steps in GRPOConfig

_reward_step_counter = {"step": 0}


def _clf_alpha() -> float:
    """Return the current classifier weight α based on the training step."""
    step = _reward_step_counter["step"]
    if step < CLF_WARMUP_STEPS:
        return 0.0
    progress = min(1.0, (step - CLF_WARMUP_STEPS) / max(1, TOTAL_STEPS - CLF_WARMUP_STEPS))
    return CLF_WEIGHT_MAX * progress


@torch.no_grad()
def translationese_reward(texts: List[str], batch_size: int = 16) -> List[float]:
    """r_t: p(HT | text) ∈ [0, 1].  Higher = more human-like translation."""
    rewards = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc   = _clf_tok(
            batch,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 256,
        ).to(DEVICE)
        probs = F.softmax(_clf_model(**enc).logits, dim=-1)
        rewards.extend(probs[:, HT_LABEL_IDX].cpu().tolist())
    return rewards


def content_reward(hypotheses: List[str], references: List[str]) -> List[float]:
    """r_c: sentence-level chrF normalized to [0, 1]."""
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        if not hyp.strip() or not ref.strip():
            rewards.append(0.0)
        else:
            rewards.append(sacrebleu.sentence_chrf(hyp, [ref]).score / 100.0)
    return rewards


def composite_reward(hypotheses: List[str], references: List[str]) -> List[float]:
    """r_final = (1 − α) · r_c + α · r_t with linear warm-up on α."""
    alpha    = _clf_alpha()
    r_c_list = content_reward(hypotheses, references)
    if alpha == 0.0:
        return r_c_list                   # skip classifier during warm-up
    r_t_list = translationese_reward(hypotheses)
    return [
        (1.0 - alpha) * r_c + alpha * r_t
        for r_c, r_t in zip(r_c_list, r_t_list)
    ]

In [ ]:
# Quick sanity test
test_hyps = ["Jornades de Voluntariat.", "This is clearly not Valencian."]
test_refs = ["Jornades de Voluntariat.", "Hola món."]

r_c = content_reward(test_hyps, test_refs)
r_t = translationese_reward(test_hyps)

print("Test rewards (step=0, warm-up, α=0):")
for i in range(len(test_hyps)):
    print(f"  [{i}] {test_hyps[i]!r}")
    print(f"       r_c={r_c[i]:.3f}  r_t={r_t[i]:.3f}")

_reward_step_counter["step"] = TOTAL_STEPS
r_final = composite_reward(test_hyps, test_refs)
print(f"\nTest rewards (step={TOTAL_STEPS}, α={_clf_alpha():.2f}):")
for i in range(len(test_hyps)):
    print(f"  [{i}] r_final={r_final[i]:.3f}")
_reward_step_counter["step"] = 0  # reset counter

Test rewards (step=0, warm-up, α=0):
  [0] 'Jornades de Voluntariat.'
       r_c=1.000  r_t=0.543
  [1] 'This is clearly not Valencian.'
       r_c=0.088  r_t=0.598

Test rewards (step=200, α=0.30):
  [0] r_final=0.863
  [1] r_final=0.241


---
## 8. GRPO Training


In [ ]:
import importlib
import transformers.models.gemma3.modeling_gemma3 as gemma3_module
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

# ── Patch: inject token_type_ids into Gemma3's causal-mask builder ─────────────
# GRPOTrainer does not pass token_type_ids, which causes a crash with Gemma 3.
# We create_causal_mask_mapping to supply zeros when missing.

importlib.reload(gemma3_module)
gemma3_module._true_original_mask_fn = gemma3_module.create_causal_mask_mapping

def _patched_mask_fn(config, input_embeds, attention_mask, cache_position,
                     past_key_values, position_ids, token_type_ids=None,
                     pixel_values=None, is_training=False,
                     is_first_iteration=None, **kwargs):
    if is_training and token_type_ids is None:
        token_type_ids = torch.zeros(
            input_embeds.shape[:2], dtype=torch.long, device=input_embeds.device
        )
    return gemma3_module._true_original_mask_fn(
        config, input_embeds, attention_mask, cache_position,
        past_key_values, position_ids, token_type_ids,
        pixel_values, is_training, is_first_iteration, **kwargs
    )

gemma3_module.create_causal_mask_mapping = _patched_mask_fn
print(" Causal-mask patch applied")

# ── Patch: inject token_type_ids into the model's forward pass ────────────────
_ModelClass = type(model)
if not getattr(_ModelClass, "_forward_patched", False):
    _ModelClass._true_original_forward = _ModelClass.forward

    def _patched_forward(self, *args, **kwargs):
        if kwargs.get("token_type_ids") is None and "input_ids" in kwargs:
            kwargs["token_type_ids"] = torch.zeros_like(kwargs["input_ids"])
        return _ModelClass._true_original_forward(self, *args, **kwargs)

    _ModelClass.forward = _patched_forward
    _ModelClass._forward_patched = True
    print(" Forward-pass patch applied")
else:
    print(" Forward-pass patch already present, skipping")

 Causal-mask patch applied
 Forward-pass patch applied


In [ ]:
# Reward wrapper for GRPOTrainer

model.train()

def grpo_reward_fn(prompts, completions, reference=None, **kwargs):
    """Strip the prompt prefix from each completion, then compute rewards."""
    clean = [c.split("model\\n")[-1].strip() for c in completions]
    _reward_step_counter["step"] += 1
    alpha = _clf_alpha()

    rewards = (
        content_reward(clean, [""] * len(clean))
        if reference is None
        else composite_reward(clean, list(reference))
    )

    mean_r = sum(rewards) / len(rewards)
    print(f"[reward] step={_reward_step_counter['step']:3d}  "          f"alpha={alpha:.3f}  mean_reward={mean_r:.4f}")
    return rewards


class SampleLoggerCallback(TrainerCallback):
    """Print a few model predictions"""

    def __init__(self, tokenizer, model, dataset, every_n_steps: int = 50, n_examples: int = 3):
        self.tokenizer  = tokenizer
        self.model      = model
        self.dataset    = dataset
        self.every_n    = every_n_steps
        self.n_examples = n_examples

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n != 0 or state.global_step == 0:
            return

        self.model.eval()
        indices = torch.randint(len(self.dataset), (self.n_examples,)).tolist()

        print(f"\n{'='*70}")
        print(f"  SAMPLES AT STEP {state.global_step}")
        print(f"{'='*70}")

        for idx in indices:
            sample = self.dataset[idx]
            inputs = self.tokenizer(
                sample["prompt"],
                return_tensors = "pt",
                truncation     = True,
                max_length     = 256,
            ).to(self.model.device)

            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens = 100,
                    do_sample      = False,
                    pad_token_id   = self.tokenizer.eos_token_id,
                )

            input_len = inputs["attention_mask"][0].sum().item()
            generated = self.tokenizer.decode(
                output_ids[0][input_len:], skip_special_tokens=True
            ).strip()

            src_text = sample["prompt"].split("\\n\\n")[-1].replace("<end_of_turn>", "").strip()
            chrf_val = sacrebleu.sentence_chrf(generated, [sample["reference"]]).score
            print(f"\n  SOURCE    : {src_text[:120]}")
            print(f"  REFERENCE : {sample['reference'][:120]}")
            print(f"  PREDICTION: {generated[:120]}")
            print(f"  chrF      : {chrf_val:.1f}")

        print(f"{'='*70}\n")
        self.model.train()


In [ ]:
# GRPO config and trainer

grpo_config = GRPOConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate               = 5e-6,
    max_steps                   = TOTAL_STEPS,
    warmup_steps                = 20,
    optim                       = "paged_adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    gradient_checkpointing      = True,
    beta                        = 0.04,
    num_generations             = 2,
    max_completion_length       = 100,
    temperature                 = 0.9,
    output_dir                  = GRPO_OUTPUT_DIR,
    logging_steps               = 1,
    save_steps                  = 20,
    seed                        = 3407,
    report_to                   = "none",
    fp16                        = False,
    bf16                        = True,
)

grpo_trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = grpo_reward_fn,
    args             = grpo_config,
    train_dataset    = grpo_dataset,
)

grpo_trainer.add_callback(
    SampleLoggerCallback(tokenizer, model, grpo_dataset, every_n_steps=50, n_examples=3)
)

In [ ]:
grpo_stats = grpo_trainer.train()

print(f"\nGRPO completado")
print(f"VRAM máxima: {round(torch.cuda.max_memory_reserved()/1e9,2)} GB")
print(f"Tiempo     : {grpo_stats.metrics['train_runtime']:.1f}s")

[reward] step= 23  alpha=0.000  mean_reward=0.8062


Step,Training Loss
1,-0.002295
2,0.003852
3,-0.189356
4,0.000019
5,0.046638
6,-0.025901
7,0.003118
8,-0.147541
9,0.017256
10,-0.091024


[reward] step= 24  alpha=0.000  mean_reward=0.9359
[reward] step= 25  alpha=0.000  mean_reward=0.6687
[reward] step= 26  alpha=0.000  mean_reward=0.8144
[reward] step= 27  alpha=0.000  mean_reward=0.7999
[reward] step= 28  alpha=0.000  mean_reward=0.8491
[reward] step= 29  alpha=0.000  mean_reward=0.9586
[reward] step= 30  alpha=0.000  mean_reward=0.6434
[reward] step= 31  alpha=0.000  mean_reward=0.6638
[reward] step= 32  alpha=0.000  mean_reward=0.7003
[reward] step= 33  alpha=0.000  mean_reward=0.7817
[reward] step= 34  alpha=0.000  mean_reward=0.5359
[reward] step= 35  alpha=0.000  mean_reward=0.8835
[reward] step= 36  alpha=0.000  mean_reward=0.8790
[reward] step= 37  alpha=0.000  mean_reward=0.4521
[reward] step= 38  alpha=0.000  mean_reward=0.7117
[reward] step= 39  alpha=0.000  mean_reward=0.5197
[reward] step= 40  alpha=0.000  mean_reward=0.9714
[reward] step= 41  alpha=0.000  mean_reward=0.8428
[reward] step= 42  alpha=0.000  mean_reward=0.8026
[reward] step= 43  alpha=0.000 

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



  EJEMPLOS EN STEP 50

  FUENTE    : ¿Seis años de alcalde -con un paréntesis de dos enmedio- no es suficiente?
<start_of_turn>model
  REFERENCIA: ¿Sis anys d'alcalde -amb un parèntesi de dos enmig- no és suficient?
  PREDICCIÓN: Sis anys d'alcalde -amb un parèntesi de dos a mitjan partit- no n'hi ha prou?
  chrF      : 64.7

  FUENTE    : La UPV quiere manifestar su máxima tristeza y dolor ante la tragedia de Campanar y pone a disposición de las autoridades
  REFERENCIA: La UPV vol manifestar la seua màxima tristesa i dolor davant la tragèdia de Campanar i posa a la disposició de les autor
  PREDICCIÓN: La UPV vol manifestar la seua màxima tristesa i dolor davant la tragèdia de Campanar i posa a la disposició de les autor
  chrF      : 99.8

  FUENTE    : El Ayuntamiento de Turís recoge alimentos para distribuir entre las personas más necesitadas
<start_of_turn>model
  REFERENCIA: L’Ajuntament de Turís recull aliments per a distribuir entre les persones més necessitades
  PREDICCIÓN:

OutOfMemoryError: CUDA out of memory. Tried to allocate 7.04 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.73 GiB is free. Including non-PyTorch memory, this process has 9.83 GiB memory in use. Of the allocated memory 8.15 GiB is allocated by PyTorch, and 1.54 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## 9. Save and Push GRPO Model


In [ ]:
# Merge LoRA weights into the base model and upload to the Hub
MERGED_DIR = GRPO_OUTPUT_DIR + "_merged"

merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to: {MERGED_DIR}")

In [ ]:
merged.push_to_hub("guerreropaula/80translategemma4b-grpo-es-va", token=HF_TOKEN)
tokenizer.push_to_hub("guerreropaula/80translategemma4b-grpo-es-va", token=HF_TOKEN)